In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A1_introduction/5_classification.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Rudiments of Machine Learning: Learning to Abstract

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 23.04.2026
+ **Authors:** [Dr. Benedikt Zönnchen](https://bzoennchen.github.io/Pages/)

In this notebook, we will teach a neural network to recognise music genres — the same challenge a human music librarian faces every day. This is a *classification task*.

⛳ **By the end of this notebook you will be able to:**
- Explore audio data
- Extract numerical features from the sound
- Build a neural network for classification
- Evaluate what it learned and what it got wrong


**⚠️ Before you start — important notes for Google Colab:**
- **Save your work first:** Go to *File → Save a copy in Drive* to save this notebook to your Google Drive. After that, Colab will auto-save your progress. You can also download your work at any time via *File → Download → Download .ipynb*❗
- **Run cells in order, from top to bottom.** Each cell can only use variables defined in cells that have already been run❗ 
- **Got an unexpected error?** Go to *Runtime → Restart session and run all* to start fresh❗ 
- **Colab disconnects after ~30 minutes of inactivity**, which erases all your variables. If this happens, go to *Runtime → Restart session and run all*. Your saved notebook in Drive is not affected❗ 


In [ ]:
# Install required libraries (run once)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'librosa', 'torch', 'torchvision', 'torchview',
                'scikit-learn', 'matplotlib', 'seaborn', 'tqdm', 'numpy', 'pandas', 'kaggle'], 
                capture_output=True)

# there are no tests for this one => no otter!

## 14 Classification of Musical Genres

So, what is the difference between regression and classicifacion. First of all, they are both *supervised learning* task, meaning that our training data contains the input as well as the expected output We know what our trained model should output for certain inputs.

At its core, the difference between *regression* and *classification* boils down to the type of "answer" you are trying to predict.

---

🗣 **Definitions**

**Regression** is used when the result you want is a number on a continuous scale. It's about measuring quantity or magnitude. Think of it as a sliding scale where any value (including decimals) is possible. For example, predicting the price of a house based on its square footage or estimating the temperature tomorrow.

**Classificiation** is used when the result is a *category* or a *label*. It's about sorting things into distinct buckets. There is no "in-between"—an item either belongs to a group or it doesn't. For example, deciding if an email is "Spam" or "Not Spam" or identifying if an image shows a "Cat" or a "Dog" or, in our case, identifying the genre of a track.

---

### 14.1 Learning is like Compression

In *machine learning* and also in *information theory* we can draw a connection between *learning* and the *compression* of information. Take a look at the following sequence of bits:

$$0101010101010101010101010101$$

There is clearly a pattern: We start with 0 and then iterate between 1 and 0. I could even write

$$01 \times 14$$

and you would probably understand that it means the same thing. Therefore we have compressed 

$$0101010101010101010101010101$$

into a "shorter" representation 

$$01 \times 14.$$

We could also see this as replacing a long answer or an observation with a short mechanism (algorithm) that **causes** the observation!
In our case writing down $01$ fourteen times **is causing** $0101010101010101010101010101$.
What we learned **is** the mechanism that explains us "the world".
In this information-theory-loaded view, Newton's physical laws of motion are compression mechanisms that we learned to explain "the world", meaning to make predictions about it!

In the case above, the compression is without a loss of information, meaning that we can reconstruct our observations when executing the mechanism.
In *machine learning* this is usually not the case which leads to the term "*abstraction*" hinted in the headline.

We usually transform our input / observation into more *abstract representations* that disregard all the small details of our input. For example, if our model classifies images into "dog", "cat", one can say that it abstracts a high-dimensional image into a single category label. 

This is also partly how our language seem to work: We say "there is a dog" and thereby compress our sensory data (of recognizing any dog) into a category "dog". And if you further think about it, there is no escape from it in our daily life. Any word and any sentence can be seen as a compression / abstract representation. The question then becomes: "Which compressions are useful or harmful?"

---

🗣 **Discourse:** This is not the only view but a useful one to understand *machine learning*. It assumes that words **represent** things "out there" and that they more or less describe reality accurately which often tends to the reasonable conclusion that we are in a process of describing reality more and more accurately i.e. that our descriptions converge towards the "really real" and the "really true".

---

What the model learns in classification is what information to keep and what information can be disregarded.
For example, it will learn to ignore the background, since the background does not (at least it should not) **cause** the label "dog" or "cat".
Of course, the **causes** can be wrong! For example, if all cats are displayed on a white background and all dogs on a black one, the model will learn that white causes cats - it will cheat.

When you look at a visualization of a *deep neural network* you will notice layers.
The input goes through these layers in sequence, meaning the output for layer $k$ is the input for layer $k+1$.
In each step a mechanism (the layer) transforms the input into an output.
All these inputs / outputs are representations of the very first input and if you want to compress information, you start with a layer with many inputs and end with a layer with only a few inputs / outputs.
By doing so, you "force" the model to come up with a good mechanism that compresses the data in such a way that it matches our observations (training data).

---

🗣 **Hint:** Not every layer necessarily has fewer units than the previous one - in fact, intermediate layers are often wider than the input. But in classification, the network must ultimately map a high-dimensional input down to a small number of class scores. So the compression happens across the full pipeline, and the training process determines what information survives that journey to the output.

---

These mechanisms can be seen as "causes" and the output of these layers are *learned* or *hidden representations* - hidden because, unlike the inputs and labels, they are not given by the data but emerge from training.

### 14.2 Powerful Packages

Our classification task is to classify music into their genre.

We will make heavy use of certain powerful `Python` packages. This should also be a lesson that there are amazing algorithms / software out there just for you to use and that these algorithms are developed by people that cared and shared.

+ `librosa`: It is a `Python` package for music and audio analysis. It provides the building blocks necessary to create music information retrieval systems.
+ `kaggle`: It is a `Python` package to use *Kaggle*, an online community + repository + challenge giver for data science.

### 14.3 Downloading the GTZAN Dataset

The **GTZAN Genre Collection** is the most widely used benchmark dataset for music genre classification. It contains:
- **1,000 audio clips**, each 30 seconds long
- **10 genres**: blues, classical, country, disco, hip-hop, jazz, metal, pop, reggae, rock
- **100 clips per genre**

It was compiled by George Tzanetakis in 2002 and has been called the "MNIST of music classification".

We use *Kaggle* which is an online community + repository + challenge giver for data science. You might want to use it in the future for your projects. So:

1. Go to `https://www.kaggle.com/`
2. Create an account
3. Go to `Settings -> API Tokens -> Generate New Token` copy it / write it down before you close the window

Now you can fill in the following information to access *Kaggle* via the *command line interface (CLI)*. If you execute the next cell the required data should be downloaded and unpacked.

⚠️ You will download 1.21 GB of data!

In [ ]:
import os
# Replace 'your_username' and 'your_key' with the actual values from your Kaggle token
os.environ['KAGGLE_USERNAME'] = "your user name"
os.environ['KAGGLE_KEY'] = "your kaggle token"

In [ ]:
# Now you can download directly
!kaggle datasets download -d andradaolteanu/gtzan-dataset-music-genre-classification

In [ ]:
# And we can unzip / unpack the files
import zipfile

# 1. Define the path to your zip file
zip_path = 'gtzan-dataset-music-genre-classification.zip'

# 2. Define where you want the files to go
extract_path = './'

# 3. Create the folder if it doesn't exist
if not os.path.exists(extract_path):
    os.makedirs(extract_path)

# 4. Unzip the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

abs_path = os.path.abspath(extract_path)
print(f"Extraction complete! Files are in the '{abs_path}' folder.")

In [ ]:
# Check it's working
DATASET_PATH = abs_path + "/data/genres_original"
genres = sorted([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))])
print(f'Found {len(genres)} genres:')
for i, g in enumerate(genres):
    n_files = len(os.listdir(os.path.join(DATASET_PATH, g)))
    print(f'  [{i:2d}] {g:<12} — {n_files} files')

### 14.4  Exploring the Audio 🎵

Before teaching a machine anything, we need to understand the data ourselves. Let's listen to and visualise a few clips.

#### 14.4.1 What does sound look like?

Audio is a **waveform** — a sequence of pressure values measured thousands of times per second (typically 22,050 Hz). We can also represent it as a **spectrogram**, which shows how the energy at different frequencies changes over time. This is much closer to how we perceive music.

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

# Pick two contrasting genres to compare
GENRE_A = 'classical'
GENRE_B = 'metal'

def load_audio_files(genre, duration=10):
    folder = os.path.join(DATASET_PATH, genre)
    files = [f for f in os.listdir(folder) if f.endswith('.wav')]
    path = os.path.join(folder, sorted(files)[0])
    audio_wave, sample_rate = librosa.load(path, duration=duration)  # load first 10 seconds
    return audio_wave, sample_rate, path

audio_wave_a, ssample_rate_a, path_a = load_audio_files(GENRE_A)
audio_wave_b, ssample_rate_b, path_b = load_audio_files(GENRE_B)

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
fig.suptitle('Waveforms and Spectrograms', fontsize=14, fontweight='bold')

for i, (y, sr, genre) in enumerate([(audio_wave_a, ssample_rate_a, GENRE_A), (audio_wave_b, ssample_rate_b, GENRE_B)]):
    # Waveform
    ax = axes[i, 0]
    librosa.display.waveshow(y, sr=sr, ax=ax, color='steelblue' if i == 0 else 'tomato')
    ax.set_title(f'{genre.capitalize()} — Waveform')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

    # Spectrogram (mel scale)
    ax = axes[i, 1]
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel', ax=ax)
    ax.set_title(f'{genre.capitalize()} — Mel Spectrogram')
    fig.colorbar(img, ax=ax, format='%+2.0f dB')

plt.tight_layout()
plt.show()

# Listen to the clips
print(f"▶ {GENRE_A.capitalize()}:")
display(Audio(audio_wave_a, rate=ssample_rate_a))
print(f"▶ {GENRE_B.capitalize()}:")
display(Audio(audio_wave_b, rate=ssample_rate_b))

---

🖍 **Exercise 14.1:** What do you notice?

---


**Solution:** Classical music tends to have a wide dynamic range (quiet passages and loud ones), while metal is dense and sustained across frequencies. These differences are what our neural network will learn to detect — but from numbers, not from listening.

#### 14.4.2 Visualising all 10 genres

Let's look at the mel spectrogram for one example from each genre side by side.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for idx, genre in enumerate(genres):
    folder = os.path.join(DATASET_PATH, genre)
    files = sorted([f for f in os.listdir(folder) if f.endswith('.wav')])
    path = os.path.join(folder, files[0])
    y, sr = librosa.load(path, duration=10, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[idx])
    axes[idx].set_title(genre.capitalize(), fontweight='bold')
    axes[idx].set_xlabel('')

plt.suptitle('One Example Per Genre — Mel Spectrograms', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 14.5 Feature Extraction 🔬

A neural network cannot work directly on audio waveforms (at least not in this notebook - that would require a *convolutional neural network (CNN)* on spectrograms). Instead, we **summarise** each clip using a set of numbers that capture its acoustic character.
We essentially **compress** the data by well-known methods before we put them into the *neural network*!

---

🖍 **Exercise 14.2:** Can you think of features of an audio file that might be important for classifying the genre? Do you think we have to "look" at the whole song?

---

*Your answer*

#### 14.5.1 Meet the MFCC

**Mel-Frequency Cepstral Coefficients (MFCCs)** are the most widely used audio features. They capture the shape of the frequency spectrum in a way that mirrors how the human ear perceives sound — emphasising differences in the mid-range frequencies, which matter most for music and speech.

---

🗣 **Hint:** If you are interested in a more elaborate explanation of the relationship between audio (in time) and energy (in frequency) you can start e.g. [here](https://bzoennchen.github.io/supercollider-book/chapters/sounddesign/math/dft.html).

---

For each clip, we compute 13 MFCCs over time and then take their **mean** and **standard deviation** across the clip, giving us 26 numbers per clip. We also add a few extra features:

| Feature | What it captures |
|---|---|
| **MFCCs** (×13) | Overall timbral texture |
| **MFCC delta** (×13) | How the timbre changes over time |
| **Chroma** (×12) | Harmonic content — which pitch classes are present |
| **Spectral centroid** | "Brightness" — where the energy sits in the spectrum |
| **Zero crossing rate** | Noisiness vs. tonality |
| **Tempo** | Rhythmic speed |

Let us visualize the MFCCs.

In [ ]:
# Visualise MFCCs for one clip
n_of_mmfc = 13
audio_wave, sample_rate = librosa.load(os.path.join(DATASET_PATH, 'jazz', sorted(os.listdir(os.path.join(DATASET_PATH, 'jazz')))[0]), duration=10)
mfccs = librosa.feature.mfcc(y=audio_wave, sr=sample_rate, n_mfcc=n_of_mmfc)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

img = librosa.display.specshow(mfccs, x_axis='time', ax=axes[0], cmap='coolwarm')
axes[0].set_title('MFCCs over time — Jazz example')
axes[0].set_ylabel('MFCC coefficient')
fig.colorbar(img, ax=axes[0])

# Show the mean and std we'll actually use as features
means = mfccs.mean(axis=1)
stds = mfccs.std(axis=1)
x = np.arange(13)
axes[1].bar(x - 0.2, means, 0.4, label='Mean', color='steelblue')
axes[1].bar(x + 0.2, stds, 0.4, label='Std Dev', color='tomato')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'MFCC {i+1}' for i in x], rotation=45)
axes[1].set_title('Mean & Std of each MFCC (our features)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from tqdm import tqdm # this is just for the visualization of the printed progress bars
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def extract_features(file_path, duration=30):
    """Extract a fixed-size feature vector from one audio file."""
    audio_wave, sample_rate = librosa.load(file_path, duration=duration, mono=True)
    features = []

    # MFCCs — mean and std of 13 coefficients
    mfcc = librosa.feature.mfcc(y=audio_wave, sr=sample_rate, n_mfcc=13)
    features += list(mfcc.mean(axis=1)) + list(mfcc.std(axis=1))

    # MFCC delta (first derivative — captures change over time)
    mfcc_delta = librosa.feature.delta(mfcc)
    features += list(mfcc_delta.mean(axis=1)) + list(mfcc_delta.std(axis=1))

    # Chroma — 12 pitch classes
    chroma = librosa.feature.chroma_stft(y=audio_wave, sr=sample_rate)
    features += list(chroma.mean(axis=1)) + list(chroma.std(axis=1))

    # Spectral centroid (brightness)
    centroid = librosa.feature.spectral_centroid(y=audio_wave, sr=sample_rate)
    features += [centroid.mean(), centroid.std()]

    # Zero crossing rate (noisiness)
    zcr = librosa.feature.zero_crossing_rate(audio_wave)
    features += [zcr.mean(), zcr.std()]

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=audio_wave, sr=sample_rate)
    features += [float(tempo)]

    return features

# Extract features for all files (this may takes a few minutes)
print('Extracting features from all audio files...')
rows = []
for genre in tqdm(genres):
    folder = os.path.join(DATASET_PATH, genre)
    files = sorted([f for f in os.listdir(folder) if f.endswith('.wav')])
    for fname in files:
        path = os.path.join(folder, fname)
        try:
            feats = extract_features(path)
            rows.append({'genre': genre, **{f'f{i}': v for i, v in enumerate(feats)}})
        except Exception as e:
            print(f'  Skipping {fname}: {e}')

# Of course we put everything in a nice pandas Dataframe
df = pd.DataFrame(rows) 
print(f'\nDone! Dataset shape: {df.shape}')
print(f'Features per clip: {df.shape[1] - 1}')
df.head(3)

In [ ]:
# (Optional) Save features so you don't have to re-extract every time
df.to_csv('gtzan_features.csv', index=False)
print('Features saved to gtzan_features.csv')

# To reload later:
# df = pd.read_csv('gtzan_features.csv')

#### 14.5.2 Visualising feature differences between genres

Before training anything, let's check visually whether our features actually separate the genres.


---
🗣 **The Principle Component Analysis (PCA)**

Because we can only really draw in 2D use a technique called *principle component analysis (PCA)* to reduce the dimensionalty from 86 to 2.
Imagine you have a 3D point cloud and you want to reduce the dimensionalty from 3 to 2.
In this case, the PCA computes the two directions with the greatest variance / spread of points and projects all points onto the plane these two vectors span.
The goal is to lose as little information and because the information "is mostly there where things are different / surprising" the PCA is one of the best **linear** compression technique. It is in fact itself a *machine learning* algorithm!


---

In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = [c for c in df.columns if c.startswith('f')]
X = df[feature_cols].values
y_labels = df['genre'].values

# Normalise features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA to 2D for visualisation
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
palette = sns.color_palette('tab10', len(genres))

for i, genre in enumerate(genres):
    mask = y_labels == genre
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=genre.capitalize(),
               alpha=0.6, s=40, color=palette[i])

ax.set_title('Feature Space — PCA Projection (2D)', fontweight='bold', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

print(f'PCA dimensionality reduction from {X_2d.shape} to 2')
print(f'The two principal components explain {sum(pca.explained_variance_ratio_)*100:.1f}% of the variance.')

---

🖍 **Exercise 14.3:** What do you see? Some genres cluster clearly (classical is often well-separated), while others overlap (blues and country, for instance). This gives us a realistic expectation: our network won't get everything right — and the confusions it makes tend to be musically interesting.

---

### 14.6 Building the Neural Network 🧠

#### 14.6.1 What is a neural networ (again)?

A **Multi-Layer Perceptron (MLP)** is a chain of layers, where each layer is a set of *neurons*. Each neuron takes a weighted sum of its inputs, adds a bias, and passes the result through a non-linear **activation function**.

```
  Input layer       Hidden layer 1     Hidden layer 2     Output layer
  (81 features)        (256 neurons)      (128 neurons)    (10 genres)

   [f0]  ──┐
   [f1]  ──┤                             
   [f2]  ──┤── [ 256 ] ──── [ 128 ] ──── [ 10 ] → softmax → genre probabilities
    ...  ──┤      ↑             ↑
   [f86] ──┘    ReLU          ReLU
```

- **ReLU** (Rectified Linear Unit): `f(x) = max(0, x)` — keeps positive values, zeroes out negative ones. This is the activation for hidden layers.
- **Softmax**: converts the final layer's values into a probability distribution (all 10 values sum to 1).
- **Dropout**: randomly switches off neurons during training. This prevents the network from memorising the training data (*overfitting*).

#### 14.6.2 How does it learn?

Training works in a loop:
1. **Forward pass** — feed a batch of clips through the network, get genre predictions
2. **Loss** — measure how wrong the predictions are (we use *cross-entropy loss*)
3. **Backward pass** — compute how much each weight contributed to the error (*backpropagation*)
4. **Update** — nudge each weight slightly in the direction that reduces the error (*gradient descent*)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Encode genre labels as integers (0–9)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_labels)
print('Genre → integer mapping:')
for i, genre in enumerate(label_encoder.classes_):
    print(f'  {i} → {genre}')

In [ ]:
# Split into training (70%), validation (15%), and test (15%) sets
# It's important to keep these three sets separate:
#   + Training set   — the network learns from this
#   + Validation set — we monitor progress during training (the network does NOT learn from it)
#   + Test set       — we measure final performance ONLY at the very end (untouched until then)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.15, stratify=y_encoded, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15 / 0.85, stratify=y_train_val, random_state=42)

print(f'Training samples:   {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Test samples:       {len(X_test)}')

# Convert to PyTorch tensors
def to_tensor(X, y):
    return TensorDataset(torch.tensor(X, dtype=torch.float32),
                         torch.tensor(y, dtype=torch.long))

train_loader = DataLoader(to_tensor(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(to_tensor(X_val, y_val),   batch_size=32)
test_loader  = DataLoader(to_tensor(X_test, y_test), batch_size=32)

In [ ]:
class MusicMLP(nn.Module):
    """
    A simple Multi-Layer Perceptron for music genre classification.

    Architecture:
        Input (n_features) → Linear → BatchNorm → ReLU → Dropout
                           → Linear → BatchNorm → ReLU → Dropout
                           → Linear → BatchNorm → ReLU → Dropout
                           → Output (n_classes)
    """
    def __init__(self, n_features, n_classes, hidden_sizes=(256, 128, 64), dropout=0.3):
        super().__init__()
        layers = []
        in_size = n_features
        for h in hidden_sizes:
            layers += [
                nn.Linear(in_size, h),
                nn.BatchNorm1d(h),  # normalise activations — speeds up training
                nn.ReLU(),
                nn.Dropout(dropout),  # randomly deactivate neurons — prevents overfitting
            ]
            in_size = h
        layers.append(nn.Linear(in_size, n_classes))  # output layer (no softmax: handled by loss fn)
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


n_features = X_train.shape[1]
n_classes  = len(genres)

print(f'Model uses {n_features} input features and predicts {n_classes} classes')

model = MusicMLP(n_features=n_features, n_classes=n_classes).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal trainable parameters: {total_params:,}')

Let us visualize the model:

In [ ]:
from torchview import draw_graph
print(X_train.shape)
with torch.no_grad():
  model_graph = draw_graph(model, input_size=X_train.shape)
model_graph.visual_graph

### 14.7 Training 🏋️

We train the network by repeatedly showing it batches of clips, measuring the error, and updating the weights. We'll track the **loss** (lower is better) and **accuracy** (higher is better) on both the training and validation sets at the end of each *epoch* (one full pass through the training data).

**Watching for overfitting:** If training accuracy keeps rising but validation accuracy stops improving (or gets worse), the network is memorising training examples rather than learning general patterns. Dropout and BatchNorm help prevent this.

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()  # activate dropout and batch norm in training mode
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()         # clear gradients from last step
        outputs = model(X_batch)      # forward pass
        loss = criterion(outputs, y_batch)  # compute loss
        loss.backward()               # backward pass (compute gradients)
        optimizer.step()              # update weights
        total_loss += loss.item() * len(y_batch)
        correct += (outputs.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()  # deactivate dropout — use full network for inference
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():  # no need to track gradients during evaluation
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * len(y_batch)
            correct += (outputs.argmax(1) == y_batch).sum().item()
            total += len(y_batch)
    return total_loss / total, correct / total

In regression we used the *mean squared error* (MSE) as our loss function.
Recall that the MSE measures how far our predicted values $\hat{y}_i$ are from the true values $y_i$:

$$L_{\text{MSE}} = \frac{1}{n} \sum_{i=1}^{n} (y_i - h_\theta(x_i))^2 = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2.$$

This works well when $y_i$ is a continuous number - a price, a temperature, a distance.
But what if $y_i \in \{0, 1\}$, i.e. our task is *classification*?
We need a loss function that speaks the language of *probabilities* rather than distances.

---

**But Why not just use MSE for classification?**

Suppose our model outputs $\hat{y}_i = p_i \in [0, 1]$, the predicted probability that the input belongs to class 1.
You *could* use the MSE, and it would technically work: if the true label is 1 and the model predicts 0.9, the squared error $(1 - 0.9)^2 = 0.01$ is small.

However, there are problems:

1. **Weak gradients for confident wrong predictions.** If we combine MSE with a sigmoid output, the gradient can become very small precisely when the model is confidently wrong - exactly when we want the strongest learning signal.
2. **No probabilistic interpretation.** The MSE does not arise from any natural probabilistic model of classification. As we will see, cross-entropy does - it falls out of *maximum likelihood estimation*.

---

Let us think about what our model is really doing.
For a single input $\mathbf{x}_i$, the model outputs a probability $p_i = f(\mathbf{x}_i) \in [0,1]$ that the label is 1.
The true label $y_i$ is either 0 or 1.

We can write the probability of observing the true label $y_i$ under the model as:

$$P(y_i \mid \mathbf{x}_i) = p_i^{y_i} \cdot (1 - p_i)^{1 - y_i}.$$

This is just the *Bernoulli distribution*. Check for yourself:

- If $y_i = 1$: $P(y_i \mid \mathbf{x}_i) = p_i$
- If $y_i = 0$: $P(y_i \mid \mathbf{x}_i) = 1 - p_i$

So the model is "good" if it assigns high probability to the true labels.

---

If we assume that our $n$ training examples are independent, then the probability of observing *all* the true labels is the product:

$$\mathcal{L} = \prod_{i=1}^{n} p_i^{y_i} \cdot (1 - p_i)^{1 - y_i}.$$

This is the *likelihood* of the data given our model.
Training the model means finding the parameters that **maximize** this likelihood - we want the model that makes our observed data most probable.

Products are awkward to work with (and numerically unstable), so we take the logarithm. Since $\log$ is monotonically increasing, maximizing the likelihood is the same as maximizing the *log-likelihood*:

$$\log \mathcal{L} = \sum_{i=1}^{n} \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right].$$

Since we typically *minimize* loss functions rather than maximize them, we flip the sign and arrive at the **binary cross-entropy loss**:

$$L_{\text{BCE}} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right].$$

That is it. Cross-entropy is nothing more than the *negative log-likelihood* of a Bernoulli model.

---

Recall our earlier idea that *learning is compression*: we look for short descriptions (mechanisms) that explain our observations.
Cross-entropy has a direct interpretation in this framework.

In information theory, the **entropy** of a distribution $q$ measures the average number of bits needed to encode outcomes drawn from $q$ using the *optimal* code for $q$ (we want to use fewer bits for encode more likely outcomes):

$$H(q) = -\sum_x q(x) \log_2 q(x).$$

The **cross-entropy** between a true distribution $q$ and a model distribution $p$ measures the average number of bits needed to encode outcomes drawn from $q$ using a code that was designed for $p$:

$$H(q, p) = -\sum_x q(x) \log_2 p(x).$$

If $p = q$, the cross-entropy equals the entropy - we are using the best possible code.
If $p \neq q$, we waste bits, because our code does not match the true distribution.
The "wasted bits" are measured by the *Kullback-Leibler divergence* $D_{\text{KL}}(q \| p) = H(q, p) - H(q) \geq 0$.

So when we minimize the binary cross-entropy loss, we are telling our model:

> "Find the probability distribution $p$ that compresses the true labels using the fewest bits."

This connects directly to our view that learning means finding a compact mechanism that explains the data.
A model with low cross-entropy has found a good "code" for the labels - it is not surprised by what it sees.
A model with high cross-entropy is a bad compressor - it wastes bits by assigning low probability to events that actually happen.

---

It is worth building some intuition for how the loss behaves.
Consider a single example where the true label is $y = 1$:

| $p$ (model's prediction) | $-\log(p)$ (loss) |
|---|---|
| 0.99 | 0.01 |
| 0.9 | 0.11 |
| 0.5 | 0.69 |
| 0.1 | 2.30 |
| 0.01 | 4.61 |

Notice that the loss grows **logarithmically** as $p \to 0$.
A confident wrong prediction ($p = 0.01$ when $y = 1$) is punished much more severely than a slightly uncertain correct prediction ($p = 0.9$ when $y = 1$).
This is exactly what we want: the model pays a heavy price for being confidently wrong.

Compare this to MSE, where the loss for $p = 0.01$ when $y = 1$ would be $(1 - 0.01)^2 = 0.98$ - a finite, bounded penalty.
Cross-entropy's unbounded penalty for confident mistakes drives the model to *never* assign near-zero probability to something that actually happens.

---

| | Regression (MSE) | Classification (Cross-Entropy) |
|---|---|---|
| **Output** | Continuous value $\hat{y} \in \mathbb{R}$ | Probability $p \in [0,1]$ |
| **Measures** | Squared distance to the true value | Negative log-probability of the true label |
| **Derived from** | Maximizing likelihood under a Gaussian noise model | Maximizing likelihood under a Bernoulli model |
| **Information-theoretic view** | — | Bits wasted by using a suboptimal code |

So the step from MSE to cross-entropy is not arbitrary.
Both loss functions arise from the same principle - *maximum likelihood estimation* - applied to different assumptions about the data.
MSE assumes continuous outputs with Gaussian noise; cross-entropy assumes binary outcomes with Bernoulli probabilities.
And through the lens of information theory, minimizing cross-entropy means learning the most efficient compression of the true labels.


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
EPOCHS        = 80       # how many times we go through the full training set
LEARNING_RATE = 1e-3     # how big a step to take when updating weights
WEIGHT_DECAY  = 1e-4     # L2 regularisation — another way to prevent overfitting
# ─────────────────────────────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss()  # standard loss using the cross-entropy for multi-class classification  !!!!!
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)  # halve lr every 20 epochs

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

print(f'Training for {EPOCHS} epochs...\n')
print(f'{"Epoch":>6}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Loss":>8}  {"Val Acc":>7}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')  # save the best checkpoint

    if epoch % 10 == 0 or epoch == 1:
        print(f'{epoch:>6}  {train_loss:>10.4f}  {train_acc*100:>8.1f}%  {val_loss:>8.4f}  {val_acc*100:>6.1f}%')

print(f'\nBest validation accuracy: {best_val_acc*100:.1f}%')

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, EPOCHS + 1)

axes[0].plot(epochs, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs, history['val_loss'],   label='Validation', color='tomato', linestyle='--')
axes[0].set_title('Loss over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train', color='steelblue')
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   label='Validation', color='tomato', linestyle='--')
axes[1].set_title('Accuracy over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

🖍 **Exercise 14.4:** What if the training accuracy is much hogher than the validation accuracy? What could you do? 

---

### 15.8 Evaluation 📊

Now we load the best model and evaluate it on the **test set** - data the network has never seen before, not even indirectly.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Load the best checkpoint
model.load_state_dict(torch.load('best_model.pt', map_location=device))

# Get all predictions on the test set
model.eval() # set the model into eval mode (e.g. we do not need to remember anythin for gradient computation)
all_preds, all_true = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch.to(device))
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

test_acc = (all_preds == all_true).mean()
print(f'Test accuracy: {test_acc*100:.1f}%')
print(f'(Chance level would be {100/n_classes:.1f}%)\n')
print(classification_report(all_true, all_preds, target_names=label_encoder.classes_))

#### 14.8.1 Visualization of the Model's Predictions

---

🖍 **Exercise 14.5:** Find out the meaning of precision, recall and f1-score and explain these terms in your own words.

---

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # normalise by row

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, fmt, title in [
    (axes[0], cm,      'd',    'Confusion Matrix (raw counts)'),
    (axes[1], cm_norm, '.2f',  'Confusion Matrix (normalised)'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=[g.capitalize() for g in label_encoder.classes_],
                yticklabels=[g.capitalize() for g in label_encoder.classes_],
                ax=ax, linewidths=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted genre')
    ax.set_ylabel('True genre')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---

🖍 **Exercise 14.6:** For what genres you get the most preceise answers from your model (according to the test data)?

---

*Your answer*

---

🖍 **Exercise 14.7:** If you have a rock song. What are the three most likely genres your model predicts (according to the test data).

---

*Your answer*

---

🖍 **Exercise 14.8:** For what genre your model predicts a wrong category most confidently?

---

*Your answer*

In [ ]:
# Per-genre accuracy bar chart
per_genre_acc = cm_norm.diagonal()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([g.capitalize() for g in label_encoder.classes_],
              per_genre_acc * 100,
              color=[plt.cm.RdYlGn(a) for a in per_genre_acc])
ax.axhline(y=test_acc * 100, color='steelblue', linestyle='--', label=f'Overall: {test_acc*100:.1f}%')
ax.set_title('Per-Genre Classification Accuracy', fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.legend()
for bar, acc in zip(bars, per_genre_acc):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{acc*100:.0f}%', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

#### 14.8.2 Interpreting the Results

Take a close look at the confusion matrix. Some questions to discuss:

- **Which genres does the network confuse most often?** Are these musically reasonable mistakes (e.g., blues and jazz) or surprising ones?
- **Which genre is hardest to classify?** Is this because the genre is internally diverse (e.g., rock spans many styles), or because it borrows from others?
- **Is our feature set sufficient?** What information does a human use to identify genres that we haven't captured in our features?
- **What would a larger or deeper network change?** Would more parameters help, or would we need more data?

### 14.9 Challenges 🔧 

#### 14.9.1 Classify your own audio file

In [ ]:
def predict_genre(file_path, model, scaler, label_encoder, top_k=3):
    """Predict the genre of a new audio file and show top-k probabilities."""
    # Extract features
    feats = extract_features(file_path)
    X_new = scaler.transform([feats])
    X_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]

    # Show results
    top_indices = probs.argsort()[::-1][:top_k]
    print(f'Predictions for: {os.path.basename(file_path)}')
    print('-' * 35)
    for i in top_indices:
        print(f'  {label_encoder.classes_[i]:<12} {probs[i]*100:5.1f}%')

    return label_encoder.classes_[probs.argmax()]


# ✏️  Replace this path with your own audio file (.wav or .mp3)
# predict_genre('your_song.wav', model, scaler, label_encoder)

If you want to go further, here are some directions to explore:

**Level 1 - Tune the network**

Modify the `MusicMLP` and training hyperparameters and observe the effect:
- Change `hidden_sizes`: try `(128, 64)` (smaller) or `(512, 256, 128)` (larger). How small can you get a model that still performs decently?
- Change `dropout`: what happens at 0.0? At 0.6? Also ask / think about what `dropout` does and why it is useful?
- Change `LEARNING_RATE`: try 1e-2 or 1e-4
- Change `EPOCHS`: does training longer always help?


**Level 2 - Feature engineering**

- Remove the delta MFCCs from `extract_features()` - how much does it hurt?
- Add **spectral bandwidth** or **spectral rolloff** from librosa and see if accuracy improves
- What if you only used 3 MFCCs instead of 13?


**Level 3 - Understand what the network learned**
- For a misclassified example, load the audio clip and listen to it. Does the network's mistake make sense?
- Try visualising the activations of the first hidden layer for classical vs. metal clips. Do they look different?


**Level 4 - Classify your own music**

Write a function that loads a new audio file, extracts features, runs it through the trained model, and returns the predicted genre with confidence scores.

---

## Summary

In this notebook you:

1. **Explored** 1,000 music clips visually and aurally - building intuition before touching any model
2. **Extracted** 87 numerical features per clip using librosa - translating sound into something a network can process
3. **Built** a Multi-Layer Perceptron with PyTorch - a fully connected network with ReLU activations, BatchNorm, and Dropout
4. **Trained** the network and monitored its learning curves - watching loss decrease and accuracy rise
5. **Evaluated** its performance with a confusion matrix - and discussed what the mistakes tell us about music and about machine learning

The key insight: a neural network does not listen to music. It learns statistical patterns in numbers derived from sound. The gap between those numbers and musical meaning is exactly where the interesting questions live.

---